In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
from datetime import date
 
CATALOG        = "clutchlytics"
BRONZE_TABLE   = f"{CATALOG}.bronze.raw_nhl_rosters"
DIM_TEAMS      = f"{CATALOG}.silver.dimTeams"
SILVER_TABLE   = f"{CATALOG}.silver.dimAthletes"
 
# Sport/league context — update when running for NFL, NBA, etc.
SPORT  = "hockey"
LEAGUE = "nhl"
 
# Initial load date — populates effective_from on first run.
# Set to the start of the current season.
# On re-runs (trade/position updates) effective_from uses ingested_at automatically.
SEASON_START = date(2025, 10, 1)
 
print(f"Source       : {BRONZE_TABLE}")
print(f"dimTeams ref : {DIM_TEAMS}")
print(f"Target       : {SILVER_TABLE}")
print(f"Sport        : {SPORT}")
print(f"League       : {LEAGUE}")
print(f"Season start : {SEASON_START}")

In [0]:
# ── READ BRONZE ───────────────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
 
bronze_df   = spark.table(BRONZE_TABLE)
ingested_at = datetime.now(timezone.utc).isoformat()
 
print(f"Bronze rows  : {bronze_df.count()}")
print(f"ingested_at  : {ingested_at}")

In [0]:
# ── JOIN TO dimTeams — RESOLVE clutch_team_id ─────────────────────────────────
 
dim_teams_df = spark.table(DIM_TEAMS).filter(F.col("league") == LEAGUE)
 
teams_ref = dim_teams_df.select(
    F.col("clutch_team_id"),
    F.col("abbreviation").alias("dim_abbreviation"),
)
 
transformed_df = (
    bronze_df
    .join(
        teams_ref,
        bronze_df.team_abbreviation == teams_ref.dim_abbreviation,
        how="left"
    )
    .select(
        F.col("athlete_id").cast("integer").alias("athlete_id"),
        F.col("clutch_team_id"),
        F.col("team_abbreviation"),
        F.lit(SPORT).alias("sport"),
        F.lit(LEAGUE).alias("league"),
        F.col("full_name"),
        F.col("display_name"),
        F.col("short_name"),
        F.col("first_name"),
        F.col("last_name"),
        F.col("position_group"),
        F.col("position_name"),
        F.col("position_abbr"),
        F.col("jersey"),
        F.col("status"),
        F.col("status_abbr"),
        F.col("age").cast("integer").alias("age"),
        F.col("date_of_birth"),
        F.col("birth_city"),
        F.col("birth_country"),
        F.col("height").cast("double").alias("height"),
        F.col("weight").cast("double").alias("weight"),
        F.col("experience_years").cast("integer").alias("experience_years"),
        F.col("shoots_catches"),
        F.col("season").cast("integer").alias("season"),
        F.lit(ingested_at).alias("ingested_at"),
        F.lit("bronze.raw_nhl_rosters").alias("source_table"),
    )
    .dropDuplicates(["athlete_id", "league"])
)
 
# Warn on unmatched team joins before proceeding
unmatched = transformed_df.filter(F.col("clutch_team_id").isNull()).count()
print(f"Transformed rows : {transformed_df.count()}")
print(f"Unmatched teams  : {unmatched}  {'<-- investigate before proceeding' if unmatched > 0 else '✓'}")
 
if unmatched > 0:
    transformed_df.filter(F.col("clutch_team_id").isNull()) \
        .select("athlete_id", "full_name", "team_abbreviation") \
        .show(truncate=False)

In [0]:
# ── FIRST LOAD vs RE-RUN DETECTION ───────────────────────────────────────────
 
table_exists = spark.catalog.tableExists(SILVER_TABLE)
 
if table_exists:
    max_id = spark.sql(f"""
        SELECT COALESCE(MAX(clutch_athlete_id), 0) AS max_id
        FROM {SILVER_TABLE}
    """).collect()[0]["max_id"]
 
    existing_leagues = spark.sql(f"""
        SELECT DISTINCT league FROM {SILVER_TABLE}
    """).rdd.flatMap(lambda x: x).collect()
 
    print(f"Table exists.")
    print(f"Max clutch_athlete_id : {max_id}")
    print(f"Existing leagues      : {existing_leagues}")
    is_first_load = LEAGUE not in existing_leagues
 
else:
    max_id           = 0
    existing_leagues = []
    is_first_load    = True
    print("Table does not exist — first load.")
 
print(f"Is first load for {LEAGUE}: {is_first_load}")

In [0]:
# ── BRANCH: FIRST LOAD ────────────────────────────────────────────────────────
# Assign clutch_athlete_id sequentially.
# effective_from = SEASON_START (not ingested_at — more accurate for initial load).
# effective_to   = null (all records open / current).
# is_current     = true for all rows.
 
if is_first_load:
 
    window_spec = Window.orderBy("clutch_team_id", "athlete_id")
 
    silver_df = (
        transformed_df
        .withColumn(
            "clutch_athlete_id",
            F.row_number().over(window_spec) + max_id
        )
        .withColumn("is_current",     F.lit(True))
        .withColumn("effective_from", F.lit(str(SEASON_START)).cast("date"))
        .withColumn("effective_to",   F.lit(None).cast("date"))
        .select(
            "clutch_athlete_id",
            "athlete_id",
            "clutch_team_id",
            "team_abbreviation",
            "sport",
            "league",
            "full_name",
            "display_name",
            "short_name",
            "first_name",
            "last_name",
            "position_group",
            "position_name",
            "position_abbr",
            "jersey",
            "status",
            "status_abbr",
            "age",
            "date_of_birth",
            "birth_city",
            "birth_country",
            "height",
            "weight",
            "experience_years",
            "shoots_catches",
            "is_current",
            "effective_from",
            "effective_to",
            "season",
            "ingested_at",
            "source_table",
        )
    )
 
    write_mode = "overwrite" if not table_exists else "append"
 
    (
        silver_df
        .write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
 
    print(f"First load complete — {silver_df.count()} rows written to {SILVER_TABLE}")

In [0]:
# ── BRANCH: RE-RUN (SCD Type 2 MERGE) ────────────────────────────────────────
# Compares incoming Bronze data against current active rows.
# Detects team changes and position changes.
# Closes old rows, inserts new rows for changed athletes.
# Simple UPDATE for non-SCD attribute changes (status, age, etc.)
 
if not is_first_load:
 
    # ── Pull current active rows for this league ──
    current_df = spark.sql(f"""
        SELECT *
        FROM {SILVER_TABLE}
        WHERE league    = '{LEAGUE}'
        AND   is_current = true
    """)
 
    print(f"Current active rows for {LEAGUE}: {current_df.count()}")
 
    # ── Identify changed athletes ──
    # Join incoming data to current active rows on athlete_id + league
    comparison_df = (
        transformed_df.alias("incoming")
        .join(
            current_df.select(
                "athlete_id",
                "league",
                "clutch_team_id",
                "position_abbr",
                "clutch_athlete_id",
            ).alias("current"),
            on=["athlete_id", "league"],
            how="left"
        )
        .withColumn(
            "team_changed",
            F.col("incoming.clutch_team_id") != F.col("current.clutch_team_id")
        )
        .withColumn(
            "position_changed",
            F.col("incoming.position_abbr") != F.col("current.position_abbr")
        )
        .withColumn(
            "scd_change",
            F.col("team_changed") | F.col("position_changed")
        )
        .withColumn(
            "is_new_athlete",
            F.col("current.athlete_id").isNull()
        )
    )
 
    # Athletes with SCD-triggering changes
    scd_athletes = comparison_df.filter(
        F.col("scd_change") == True
    )
 
    # Brand new athletes not in current table (e.g. callups, new signings)
    new_athletes = comparison_df.filter(
        F.col("is_new_athlete") == True
    )
 
    # Athletes with only non-SCD changes (status, age, etc.) 
    update_only = comparison_df.filter(
        (F.col("scd_change") == False) &
        (F.col("is_new_athlete") == False)
    )
 
    print(f"\nChange detection:")
    print(f"  SCD changes (team/position) : {scd_athletes.count()}")
    print(f"  New athletes                : {new_athletes.count()}")
    print(f"  Update-only (no SCD)        : {update_only.count()}")
 
    if scd_athletes.count() > 0:
        print("\n  SCD changes detected:")
        scd_athletes.select(
            "incoming.full_name",
            "incoming.team_abbreviation",
            "incoming.position_abbr",
            "current.clutch_team_id",
            "team_changed",
            "position_changed"
        ).show(truncate=False)
 
    # ── Step 1: Close out old SCD rows ──
    # Set is_current = false and effective_to = ingested_at
    if scd_athletes.count() > 0:
 
        changed_ids = [
            row["clutch_athlete_id"]
            for row in scd_athletes.select("current.clutch_athlete_id").collect()
        ]
        changed_ids_str = ", ".join(str(i) for i in changed_ids)
 
        spark.sql(f"""
            UPDATE {SILVER_TABLE}
            SET
                is_current   = false,
                effective_to = CAST('{ingested_at}' AS DATE)
            WHERE clutch_athlete_id IN ({changed_ids_str})
            AND   is_current = true
        """)
 
        print(f"\nStep 1 complete — closed {len(changed_ids)} old SCD rows.")
 
    # ── Step 2: Insert new rows for SCD changes + new athletes ──
    athletes_to_insert = scd_athletes.unionByName(
        new_athletes, allowMissingColumns=True
    ) if new_athletes.count() > 0 else scd_athletes
 
    if athletes_to_insert.count() > 0:
 
        # Get new max_id after potential new league appends
        new_max_id = spark.sql(f"""
            SELECT COALESCE(MAX(clutch_athlete_id), 0) AS max_id
            FROM {SILVER_TABLE}
        """).collect()[0]["max_id"]
 
        window_spec = Window.orderBy(
            F.col("incoming.clutch_team_id"),
            F.col("incoming.athlete_id")
        )
 
        new_rows_df = (
            athletes_to_insert
            .select(
                F.col("incoming.athlete_id").alias("athlete_id"),
                F.col("incoming.clutch_team_id").alias("clutch_team_id"),
                F.col("incoming.team_abbreviation").alias("team_abbreviation"),
                F.col("incoming.sport").alias("sport"),
                F.col("incoming.league").alias("league"),
                F.col("incoming.full_name").alias("full_name"),
                F.col("incoming.display_name").alias("display_name"),
                F.col("incoming.short_name").alias("short_name"),
                F.col("incoming.first_name").alias("first_name"),
                F.col("incoming.last_name").alias("last_name"),
                F.col("incoming.position_group").alias("position_group"),
                F.col("incoming.position_name").alias("position_name"),
                F.col("incoming.position_abbr").alias("position_abbr"),
                F.col("incoming.jersey").alias("jersey"),
                F.col("incoming.status").alias("status"),
                F.col("incoming.status_abbr").alias("status_abbr"),
                F.col("incoming.age").alias("age"),
                F.col("incoming.date_of_birth").alias("date_of_birth"),
                F.col("incoming.birth_city").alias("birth_city"),
                F.col("incoming.birth_country").alias("birth_country"),
                F.col("incoming.height").alias("height"),
                F.col("incoming.weight").alias("weight"),
                F.col("incoming.experience_years").alias("experience_years"),
                F.col("incoming.shoots_catches").alias("shoots_catches"),
                F.col("incoming.season").alias("season"),
                F.col("incoming.ingested_at").alias("ingested_at"),
                F.col("incoming.source_table").alias("source_table"),
            )
            .withColumn(
                "clutch_athlete_id",
                F.row_number().over(window_spec) + new_max_id
            )
            .withColumn("is_current",     F.lit(True))
            .withColumn("effective_from", F.lit(ingested_at).cast("date"))
            .withColumn("effective_to",   F.lit(None).cast("date"))
            .select(
                "clutch_athlete_id", "athlete_id", "clutch_team_id",
                "team_abbreviation", "sport", "league", "full_name",
                "display_name", "short_name", "first_name", "last_name",
                "position_group", "position_name", "position_abbr",
                "jersey", "status", "status_abbr", "age", "date_of_birth",
                "birth_city", "birth_country", "height", "weight",
                "experience_years", "shoots_catches", "is_current",
                "effective_from", "effective_to", "season",
                "ingested_at", "source_table",
            )
        )
 
        (
            new_rows_df
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(SILVER_TABLE)
        )
 
        print(f"Step 2 complete — inserted {new_rows_df.count()} new SCD rows.")
 
    # ── Step 3: UPDATE non-SCD attribute changes ──
    # Status, age, weight, jersey — update in place on the current row.
    if update_only.count() > 0:
 
        update_only.select(
            F.col("incoming.athlete_id").alias("athlete_id"),
            F.col("incoming.league").alias("league"),
            F.col("incoming.status").alias("status"),
            F.col("incoming.status_abbr").alias("status_abbr"),
            F.col("incoming.age").alias("age"),
            F.col("incoming.height").alias("height"),
            F.col("incoming.weight").alias("weight"),
            F.col("incoming.jersey").alias("jersey"),
            F.col("incoming.ingested_at").alias("ingested_at"),
        ).createOrReplaceTempView("non_scd_updates")
 
        spark.sql(f"""
            MERGE INTO {SILVER_TABLE} AS target
            USING non_scd_updates AS source
            ON  target.athlete_id = source.athlete_id
            AND target.league     = source.league
            AND target.is_current = true
            WHEN MATCHED THEN UPDATE SET
                target.status       = source.status,
                target.status_abbr  = source.status_abbr,
                target.age          = source.age,
                target.height       = source.height,
                target.weight       = source.weight,
                target.jersey       = source.jersey,
                target.ingested_at  = source.ingested_at
        """)
 
        print(f"Step 3 complete — updated {update_only.count()} non-SCD rows.")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
print("── Current active roster ──")
spark.sql(f"""
    SELECT
        clutch_athlete_id,
        athlete_id,
        full_name,
        team_abbreviation,
        position_abbr,
        is_current,
        effective_from,
        effective_to
    FROM {SILVER_TABLE}
    WHERE league     = '{LEAGUE}'
    AND   is_current = true
    ORDER BY team_abbreviation, position_abbr, last_name
""").show(50, truncate=False)
 
print("── Full history (all rows including closed) ──")
spark.sql(f"""
    SELECT
        clutch_athlete_id,
        athlete_id,
        full_name,
        team_abbreviation,
        position_abbr,
        is_current,
        effective_from,
        effective_to
    FROM {SILVER_TABLE}
    WHERE league = '{LEAGUE}'
    ORDER BY athlete_id, effective_from
""").show(50, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                AS total_rows,
        COUNT(CASE WHEN is_current = true  THEN 1 END)         AS current_rows,
        COUNT(CASE WHEN is_current = false THEN 1 END)         AS historical_rows,
        COUNT(DISTINCT athlete_id)                             AS unique_athletes,
        COUNT(DISTINCT clutch_team_id)                         AS teams_represented,
        COUNT(CASE WHEN position_abbr = 'G'  THEN 1 END)       AS goalies,
        COUNT(CASE WHEN position_abbr = 'D'  THEN 1 END)       AS defensemen,
        COUNT(CASE WHEN position_abbr NOT IN ('G','D')
                        THEN 1 END)                            AS forwards,
        COUNT(CASE WHEN clutch_team_id IS NULL
                    AND is_current = true  THEN 1 END)         AS unmatched_teams,
        COUNT(CASE WHEN effective_to IS NULL
                    AND is_current = false THEN 1 END)         AS unclosed_rows,
        MIN(clutch_athlete_id)                                 AS min_id,
        MAX(clutch_athlete_id)                                 AS max_id
    FROM {SILVER_TABLE}
    WHERE league = '{LEAGUE}'
""")
 
print(f"Sanity checks ({LEAGUE}):")
checks.show(truncate=False)
